# WS 11.1: Three Classifiers, Same Exam

In WS 12.1 you trained a decision tree on a train/test split, built a confusion matrix, and computed precision and recall on data the model had never seen. You also saw that deeper trees can overfit — doing well on training data but worse on new data.

Today you'll meet two more classifiers: **logistic regression** and a **neural network**. The big idea: the same train-predict-evaluate steps work for every classifier.

> **I will not use AI tools on this worksheet.**
>
> **Name:** \_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_

### Setup

Run this cell to load libraries and a helper function.

In [ ]:
#@title Setup — run this cell
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def get_metrics(actual, predicted):
    """Print and return accuracy, precision, and recall."""
    cm = pd.crosstab(actual, predicted)
    tp = int(cm.iloc[1, 1]) if cm.shape == (2, 2) else 0
    fp = int(cm.iloc[0, 1]) if cm.shape == (2, 2) else 0
    fn = int(cm.iloc[1, 0]) if cm.shape == (2, 2) else 0
    accuracy = (np.array(predicted) == np.array(actual)).mean()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    return accuracy, precision, recall

---

## Part 1: Three Models, One Framework

Today you'll use the same Taiwan credit dataset from WS 12.1 — 30,000 credit card customers, where `default = 1` means the customer failed to pay their next bill.

Last time you used four features. This time, you'll give the models all 23 features in the dataset. More information to work with — but does more information always mean better predictions?

### The data

In [ ]:
#@title Load data — run this cell
credit = pd.read_csv("https://raw.githubusercontent.com/statisfactions/QRAI-materials/main/data/taiwan_credit.csv")
features = credit.columns.drop("default").tolist()
credit.head()

The `features` list above holds every column name except `default`. You can use it whenever you need the 23 feature columns.

### Splitting and scaling

The same idea from WS 12.1 applies: train on one part of the data, test on a part the model never sees.

> **Reminder — the train/test split pattern from WS 12.1:**
>
> ```python
> X = credit[_____]
> y = credit["default"]
>
> X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=_____)
> ```

**Exercise 1.1:** Split the data into training and test sets. Use `test_size=0.2`. Use the `features` list for `X`.

In [ ]:
# Your code here

*Training set: \_\_\_\_ customers. Test set: \_\_\_\_ customers.*

Now, a new step before training. Decision trees compare features by asking yes/no threshold questions, so the size of the numbers does not affect the result. But logistic regression and neural networks combine features using math — adding and multiplying. If one feature goes from 0 to 900,000 (like credit limit in NT dollars) and another goes from 0 to 8 (like months late on payment), the big-number feature will take over the calculation and hide the small one.

**Scaling** fixes this. You already know mean and standard deviation from the climate change unit. Scaling takes each feature, subtracts its mean, and divides by its standard deviation. Every feature ends up centered at 0 with a spread of 1. The shape of the data does not change — just the center and scale.

We scale using the training data only, then use the same scaling on the test set. This keeps the test set truly unseen: nothing from the test set is used to prepare the data.

> **Scaling pattern:**
>
> ```python
> scaler = StandardScaler()
> X_train_scaled = scaler.fit_transform(_____)
> X_test_scaled = scaler.transform(_____)
> ```
>
> `fit_transform` learns the mean and spread from the training data and scales it in one step. `transform` uses that same scaling — without re-learning — on new data.

**Exercise 1.2:** Scale the training and test sets.

In [ ]:
# Your code here

### Meet the three models

Today you'll train three classifiers. Each one learns in a different way, but all three use the same pattern:

1. **Decision tree** — asks a list of yes/no questions about the features. You have used this since WS 10.2.

2. **Logistic regression** — combines all the features into one score using a formula, then uses a cutoff to pick a class. Think of it as drawing a line between the two groups. We will not look inside the formula — just `.fit()` and `.predict()`.

3. **Neural network** — passes the features through layers of connected nodes, each with a number the model learns. We will look at how neural networks work in a later worksheet. For now, it is another black box with the same steps.

### Model 1: Decision tree

The train/predict pattern is the same as WS 12.1, but with two new pieces: the `get_metrics` helper and multiple assignment.

`get_metrics` takes the actual labels and the model's predictions, prints accuracy, precision, and recall, and *also returns all three as values you can save*. You save all three at once using multiple assignment — the same idea as unpacking a pair, but with three variables:

> **Reminder — the sklearn train/predict/evaluate pattern:**
>
> ```python
> model = SomeClassifier(...)
> model.fit(X_train, y_train)
>
> train_pred = model.predict(X_train)
> test_pred  = model.predict(X_test)
>
> # Training set
> train_acc, train_prec, train_rec = get_metrics(y_train, train_pred)
> # Test set
> test_acc, test_prec, test_rec = get_metrics(y_test, test_pred)
> ```
>
> `get_metrics` prints accuracy, precision, and recall for you — you do not need to print them again.

**Exercise 1.3:** Train a `DecisionTreeClassifier` with `max_depth=3` on the **scaled** training set. Get predictions on both the training set and the [scaled] test set. Compute and save metrics for both.

**Note:** Name your model `tree`, your predictions `tree_train_pred` and `tree_test_pred`, and your saved metric variables with the `tree_` prefix (e.g., `tree_train_acc`, `tree_train_prec`, `tree_train_rec`, `tree_test_acc`, `tree_test_prec`, `tree_test_rec`). The summary table below will use these names.

In [ ]:
# Your code here

### Model 2: Logistic regression

The steps are the same. Only the model creation line changes.

For logistic regression, the creation line is:

```python
LogisticRegression(max_iter=1000)
```

`max_iter=1000` gives the model enough time to find a good answer. (With a large dataset like this one, the default of 100 steps is often not enough.)

**Exercise 1.4:** Train a logistic regression model on the scaled training set. Follow the same pattern as Exercise 1.3. Use `lr` as your variable prefix (`lr`, `lr_train_pred`, `lr_test_pred`, `lr_train_acc`, etc.).

In [ ]:
# Your code here

### Model 3: Neural network

One more time, same pattern. For the neural network, the creation line is:

```python
MLPClassifier(hidden_layer_sizes=(50,), max_iter=500)
```

`hidden_layer_sizes=(50,)` means one layer of 50 connected nodes between the input and the output. `MLP` stands for "multi-layer perceptron" — a type of neural network. We will look inside later. For now: same pattern, new model.

**Exercise 1.5:** Train a neural network on the scaled training set. Follow the same pattern. Use `nn` as your variable prefix (`nn`, `nn_train_pred`, `nn_test_pred`, `nn_train_acc`, etc.).

In [ ]:
# Your code here

### Comparing all three

In [ ]:
#@title Part 1 results table — run this cell
results_1 = pd.DataFrame({
    "Model": ["Decision Tree", "Logistic Regression", "Neural Network"],
    "Train Acc":  [round(tree_train_acc, 3),  round(lr_train_acc, 3),  round(nn_train_acc, 3)],
    "Test Acc":   [round(tree_test_acc, 3),   round(lr_test_acc, 3),   round(nn_test_acc, 3)],
    "Train Prec": [round(tree_train_prec, 3), round(lr_train_prec, 3), round(nn_train_prec, 3)],
    "Test Prec":  [round(tree_test_prec, 3),  round(lr_test_prec, 3),  round(nn_test_prec, 3)],
    "Train Rec":  [round(tree_train_rec, 3),  round(lr_train_rec, 3),  round(nn_train_rec, 3)],
    "Test Rec":   [round(tree_test_rec, 3),   round(lr_test_rec, 3),   round(nn_test_rec, 3)],
})
results_1

**Exercise 1.6:** Look at the Train Acc and Test Acc columns.

- For each model, is training accuracy higher or lower than test accuracy? What does that gap mean?
- Which model has the biggest gap?

*Your answer here.*

---

## Part 2: More Capacity

In Part 1, every model had limits: the decision tree could only ask 3 questions, the neural network had just 50 nodes in one layer.

What happens when we take those limits off — give each model more **capacity** to find patterns? More capacity means the model *could* learn more. But it also means more room to memorize instead of learning.

You will train three new versions of the models on the same 23 features and the same train/test split as Part 1. The only thing that changes is how much each model is allowed to do.

### Decision tree — no depth limit

**Exercise 2.1:** Train a new decision tree with no depth limit. Leave out `max_depth` entirely (or set it to `None`). Call it `tree2`. Compute metrics on both the training and test sets. Use `tree2_` as your variable prefix.

> **Hint:** You can copy your code from Exercise 1.3 and change just one thing.

In [ ]:
# Your code here

### Logistic regression — with interactions

Standard logistic regression looks at each feature by itself. But real patterns are often about *combinations*: a customer with a high credit limit who is also behind on payments might act differently than someone with a high limit who always pays on time.

We can give logistic regression access to pairs of features multiplied together — called **interactions**. With 23 features, that adds hundreds of new terms for the model to work with.

Run the cell below to create the interaction features. You do not need to understand the code — just notice how many features the bigger version has.

In [ ]:
#@title Add interaction features — run this cell
from sklearn.preprocessing import PolynomialFeatures
_poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_inter = _poly.fit_transform(X_train_scaled)
X_test_inter = _poly.transform(X_test_scaled)
print(f"Original features:    {X_train_scaled.shape[1]}")
print(f"With interactions:    {X_train_inter.shape[1]}")

**Exercise 2.2:** Train a new logistic regression model (`lr2`) on `X_train_inter`. Get predictions on `X_train_inter` and `X_test_inter`. Compute metrics for both. Use `lr2_` as your variable prefix.

> **Note:** The model creation line is the same — `LogisticRegression(max_iter=1000)`. Only the data changes.

In [ ]:
# Your code here

### Neural network — bigger network

**Exercise 2.3:** Train a bigger neural network (`nn2`) using:

```python
MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500)
```

This network has two layers instead of one: 100 nodes in the first layer, 50 in the second. Compute metrics on both sets. Use `nn2_` as your variable prefix.

In [ ]:
# Your code here

### How does capacity change things?

In [ ]:
#@title Part 2 results table — run this cell
results_2 = pd.DataFrame({
    "Model": ["Decision Tree (no limit)", "Logistic Reg. (interactions)", "Neural Network (larger)"],
    "Train Acc":  [round(tree2_train_acc, 3),  round(lr2_train_acc, 3),  round(nn2_train_acc, 3)],
    "Test Acc":   [round(tree2_test_acc, 3),   round(lr2_test_acc, 3),   round(nn2_test_acc, 3)],
    "Train Prec": [round(tree2_train_prec, 3), round(lr2_train_prec, 3), round(nn2_train_prec, 3)],
    "Test Prec":  [round(tree2_test_prec, 3),  round(lr2_test_prec, 3),  round(nn2_test_prec, 3)],
    "Train Rec":  [round(tree2_train_rec, 3),  round(lr2_train_rec, 3),  round(nn2_train_rec, 3)],
    "Test Rec":   [round(tree2_test_rec, 3),   round(lr2_test_rec, 3),   round(nn2_test_rec, 3)],
})
results_2

To compare the two tables side by side:

In [ ]:
#@title All results — run this cell
print("=== Part 1: Limited capacity ===")
display(results_1)
print("=== Part 2: More capacity ===")
display(results_2)

**Exercise 2.4:** Compare Train Acc and Test Acc for each model. A large gap means the model performs much better on data it trained on than on new data — a sign of overfitting.

- Which model overfits the most?
- Which model overfits the least?
- How does this connect to the reward-hacking discussion from the AI unit?

*Your answer here.*

**Exercise 2.5:** Compare your Part 1 and Part 2 results. Did more capacity improve test-set performance? For which models and which metrics?

*Your answer here.*

**Exercise 2.6:** The bank asks you to recommend one model — from either Part 1 or Part 2. They want to catch as many defaulters as possible (high recall) without too many false alarms (decent precision), and they want a model that will still work well on new customers.

Which model would you recommend? What trade-offs are you making?

*Your answer here.*

---

*Worksheet created by Ethan C. Brown in collaboration with Claude Code.*